# Loops and Conditionals for Batch Thinking
### 1.5 — Foundations · Beginner · 12–15 min

Labs rarely measure one sample. The moment you have a tray of them, you need to do
the *same* steps to each one — without copy-pasting a calculation twenty times and
hand-editing each. That's what a loop is for. And once you're processing a batch, you
want the computer to **flag** the samples that fall outside spec, so your eyes go
straight to what matters.

> **One idea to hold onto:** write the step once; let the loop visit every sample, and
> let an `if` decide which results need your attention.

**By the end of this notebook you will be able to:**

1. Apply the same calculation to every sample with a `for` loop.
2. Classify each result against spec limits with `if` / `elif` / `else`.
3. Collect the outcomes into a small table you can scan at a glance.

## 1. The same calculation for every sample

A batch of samples: each has an ID and a blank-corrected absorbance. We convert each
to concentration with Beer–Lambert — the identical step, applied to all of them.

In [ ]:
sample_ids = ["S-01", "S-02", "S-03", "S-04", "S-05"]
absorbance = [0.451, 0.902, 0.121, 0.660, 0.405]   # blank-corrected, AU

epsilon = 6200.0       # L / (mol * cm)
path_length_cm = 1.0

# Visit each (id, absorbance) pair and apply the same conversion.
concentrations_umol_L = []
for sid, ab in zip(sample_ids, absorbance):
    conc = (ab / (epsilon * path_length_cm)) * 1e6   # mol/L -> umol/L
    concentrations_umol_L.append(conc)
    print(f"{sid}: {ab:.3f} AU -> {conc:6.2f} umol/L")

The loop body is written **once**. Whether the tray holds 5 samples or 96, the code is
the same — you'd only change the data. That's the leap from a calculator to an
analysis.

## 2. Flagging with if / elif / else

Suppose the method's acceptance window is 40–120 umol/L. We want each sample labelled
**low**, **pass**, or **high** so out-of-spec results announce themselves.

In [ ]:
spec_low_umol_L = 40.0
spec_high_umol_L = 120.0

flags = []
for sid, conc in zip(sample_ids, concentrations_umol_L):
    if conc < spec_low_umol_L:
        flag = "LOW"
    elif conc > spec_high_umol_L:
        flag = "HIGH"
    else:
        flag = "pass"
    flags.append(flag)
    print(f"{sid}: {conc:6.2f} umol/L  ->  {flag}")

The `if` checks one condition, `elif` ("else if") checks another only if the first
failed, and `else` catches everything left. Exactly one branch runs per sample.

## 3. Count, then summarize as a table

Two quick wins. First, count how many samples need attention. Then assemble the
results into a small pandas table — one row per sample — which is far easier to scan
than scrolled print output.

In [ ]:
n_out_of_spec = sum(1 for f in flags if f != "pass")
print("out-of-spec samples:", n_out_of_spec, "of", len(sample_ids))

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "sample_id": sample_ids,
    "concentration_umol_L": [round(c, 2) for c in concentrations_umol_L],
    "flag": flags,
})
results

A glance at the `flag` column tells you where to look — the whole point of letting the
computer do the checking. (Selecting just the flagged rows from a table like this is a
Track 2 skill, with pandas.)

## Key Takeaways

- A `for` loop applies one written-once step to **every** item in a batch.
- `if` / `elif` / `else` turns a number into a **decision** (low / pass / high).
- Collecting results into a small table makes a batch scannable at a glance.

## Practical Checklist

- [ ] Write the per-sample calculation once, inside the loop.
- [ ] Pair IDs with values (`zip`) so results never get mismatched.
- [ ] Define spec limits as named constants, not magic numbers in the `if`.
- [ ] Summarize outcomes (a count, or a table) rather than eyeballing raw output.

## Common Mistakes

- Copy-pasting a calculation per sample instead of looping — errors creep into the copies.
- Off-by-one or mismatched lists, so the wrong value gets the wrong ID.
- Overlapping `if` conditions where you assume only one runs — order them so the
  branches are mutually exclusive.

## Next Lesson

**1.6 — Functions: Writing a Step Once and Reusing It.** The loop reused a step within
one notebook; a function lets you reuse the same defined method everywhere.